# Démo Google Colab — Détection d'anomalies

Ce notebook charge les CSV du kit, exécute les contrôles, puis exporte `alerts_output.csv`.

In [ ]:
!pip -q install pandas

In [ ]:
from google.colab import files
import io, json
from pathlib import Path
import pandas as pd

## 1) Téléverser les fichiers CSV et le JSON de configuration

In [ ]:
uploaded = files.upload()
list(uploaded.keys())

In [ ]:
frames = {}
for name, content in uploaded.items():
    if name.endswith('.csv'):
        frames[name] = pd.read_csv(io.BytesIO(content))
    elif name.endswith('.json'):
        thresholds = json.loads(content.decode('utf-8'))
frames.keys(), thresholds

## 2) Détection

In [ ]:
SEVERITY_SCORE = {"LOW": 1, "MEDIUM": 2, "HIGH": 3, "CRITICAL": 4}

def make_alert(alert_id, alert_type, severity, entity_id, source, rule, justification, recommended_action, owner_role):
    return {
        "alert_id": alert_id,
        "alert_type": alert_type,
        "severity": severity,
        "priority_score": SEVERITY_SCORE[severity],
        "entity_id": entity_id,
        "source_system": source,
        "rule_name": rule,
        "justification": justification,
        "status": "NEW",
        "recommended_action": recommended_action,
        "owner_role": owner_role,
        "detected_at": "2026-04-22"
    }

students = frames["students.csv"]
payments = frames["payments.csv"]
grades = frames["grades.csv"]
finance = frames["finance_transactions.csv"]

cost_centers = {"INFO":"CC-INFO","FIN":"CC-FIN","RH":"CC-RH","MKT":"CC-MKT"}

alerts = []
counter = 1

dup = students.groupby("national_id").filter(lambda x: len(x) > 1)
for national_id, group in dup.groupby("national_id"):
    alerts.append(make_alert(f"ALT{counter:04d}","DUPLICATE_REGISTRATION","HIGH",", ".join(group["student_id"].astype(str).tolist()),
                             "SCOLARITE","duplicate_national_id",f"{len(group)} inscriptions partagent le national_id {national_id}.",
                             "Vérifier et fusionner/clôturer les doublons.","MOA Scolarité"))
    counter += 1

bad_paid = payments[(payments["payment_status"]=="PAID") & ((payments["receipt_id"].fillna("")=="") | (payments["amount"]<=0))]
for _, row in bad_paid.iterrows():
    alerts.append(make_alert(f"ALT{counter:04d}","PAID_WITHOUT_TRACE","HIGH",row["payment_id"],
                             "FINANCE","paid_without_receipt_or_zero_amount",
                             f"Paiement {row['payment_id']} sans trace cohérente.","Vérifier justificatif et saisie.","MOA Finance"))
    counter += 1

bad_grades = grades[grades["grade"].isna() | (grades["grade"]<0) | (grades["grade"]>20)]
for _, row in bad_grades.iterrows():
    sev = "MEDIUM" if pd.isna(row["grade"]) else "HIGH"
    alerts.append(make_alert(f"ALT{counter:04d}","GRADE_ANOMALY",sev,row["student_id"],
                             "SCOLARITE","missing_or_invalid_grade",
                             f"Anomalie de note sur {row['course_code']}.","Vérifier le PV et republier.","MOA Pédagogie"))
    counter += 1

orphan = payments[~payments["student_id"].isin(students["student_id"])]
for _, row in orphan.iterrows():
    alerts.append(make_alert(f"ALT{counter:04d}","ORPHAN_PAYMENT","HIGH",row["payment_id"],
                             "FINANCE","payment_without_student",
                             f"Paiement orphelin {row['payment_id']}.","Corriger l'identifiant ou annuler.","MOE Data / Finance"))
    counter += 1

student_dept = students.drop_duplicates("student_id")[["student_id","department_code"]]
cross = finance.merge(student_dept, on="student_id", how="left")
cross["expected_cc"] = cross["department_code"].map(cost_centers)
cross_gap = cross[(cross["department_code"].isna()) | (cross["expected_cc"].fillna("CC-UNKNOWN") != cross["cost_center"])]
for _, row in cross_gap.iterrows():
    alerts.append(make_alert(f"ALT{counter:04d}","CROSS_SYSTEM_GAP","HIGH",row["transaction_id"],
                             "FINANCE/SCOLARITE/RH","cost_center_mismatch",
                             f"Écart centre de coût pour {row['transaction_id']}.","Corriger le mapping et l'interface.","MOE Data"))
    counter += 1

unusual = finance[finance["amount"] > thresholds["high_amount_threshold"]]
for _, row in unusual.iterrows():
    alerts.append(make_alert(f"ALT{counter:04d}","UNUSUAL_TRANSACTION","CRITICAL",row["transaction_id"],
                             "FINANCE","high_amount_threshold",
                             f"Montant inhabituel {row['amount']} XOF.","Bloquer et rapprocher.","MOA Finance"))
    counter += 1

dups_tx = finance.groupby(["student_id","transaction_date","amount"]).filter(lambda x: len(x) >= thresholds["duplicate_transaction_same_day_count"])
for _, grp in dups_tx.groupby(["student_id","transaction_date","amount"]):
    alerts.append(make_alert(f"ALT{counter:04d}","UNUSUAL_TRANSACTION","CRITICAL",", ".join(grp["transaction_id"].astype(str).tolist()),
                             "FINANCE","duplicate_same_day_transaction",
                             f"{len(grp)} transactions dupliquées.", "Vérifier et annuler les doublons.","MOE Data / Finance"))
    counter += 1

alerts_df = pd.DataFrame(alerts).sort_values(["priority_score","alert_type"], ascending=[False, True]).reset_index(drop=True)
alerts_df.head(20)

## 3) Export

In [ ]:
alerts_df.to_csv("alerts_output.csv", index=False, encoding="utf-8-sig")
files.download("alerts_output.csv")